# int4 decode: fused Q Hadamard (Triton) vs unfused (`triton_backend` style)

Matches `triton_backend.py` for **int4** when `HADAMARD=1` and `SGLANG_FUSE_HADAMARD_INT4_KV=1`:

- **Reference:** `hadamard_transform(q / sqrt(order))` then `decode_attention_fwd_quantized(..., fuse_q_hadamard=False)`.
- **Fused:** raw `q` and `fuse_q_hadamard=True` — FWHT runs inside **`_fwd_kernel_stage1_quant_int4`** (MHA) or **`_fwd_grouped_kernel_stage1_quant_int4`** (GQA).

**Requirements:** CUDA GPU, `triton`, `fast_hadamard_transform`, and `sglang` importable (`pip install -e ./python` or `PYTHONPATH` to `python/`).

**Note:** Fused FWHT uses fp32 butterfly + cast back to `q` dtype; the CUDA path is bf16 Hadamard. Expect small numeric drift; we report cosine similarity and max abs error on `o`.

In [1]:
from __future__ import annotations

import math
import os
import sys

import torch
import triton.testing

REPO_ROOT = os.environ.get("SGLANG_REPO_ROOT", os.getcwd())
PY_ROOT = os.path.join(REPO_ROOT, "python")
if os.path.isdir(PY_ROOT) and PY_ROOT not in sys.path:
    sys.path.insert(0, PY_ROOT)

assert torch.cuda.is_available(), "CUDA required"
DEVICE = torch.device("cuda")
torch.manual_seed(0)

from fast_hadamard_transform import hadamard_transform

from sglang.srt.layers.attention.triton_ops.decode_attention import (
    decode_attention_fwd_quantized,
)
from sglang.srt.mem_cache.kv_quant_kernels import quantized_set_kv_int4_triton

## 1. Sanity: unfused vs fused Q Hadamard (GQA + MHA)

Same `q` and int4 K/V for both paths per layout. `head_dim` must satisfy `validate_hadamard_order_for_kv_fuse` (power-of-two order, `head_dim % order == 0`; no DPE in these synthetic buffers).

In [2]:
def apply_triton_backend_style_q_hadamard(
    q: torch.Tensor, hadamard_order: int
) -> torch.Tensor:
    """Mirror `triton_backend.py`: view (…, head_dim) → (…, head_dim//order, order), transform, restore."""
    q = q.contiguous()
    orig = q.shape
    v = q.view(*orig[:-1], orig[-1] // hadamard_order, hadamard_order)
    v = hadamard_transform(v / math.sqrt(hadamard_order))
    return v.view(orig)


def build_gqa_int4_decode_buffers(
    batch: int,
    seq_len: int,
    num_q_heads: int,
    num_kv_heads: int,
    head_dim: int,
    head_dim_v: int | None = None,
    dtype: torch.dtype = torch.bfloat16,
    max_kv_splits: int = 8,
    kv_splits_val: int = 4,
):
    """Allocate q, int4 K/V cache, indptr/indices, logits/lse, o — grouped (GQA) decode."""
    assert num_q_heads % num_kv_heads == 0
    dv = head_dim if head_dim_v is None else head_dim_v
    total_tokens = batch * seq_len
    sm_scale = 1.0 / (head_dim**0.5)

    q = torch.randn(batch, num_q_heads, head_dim, device=DEVICE, dtype=dtype)

    k_bf16 = torch.randn(total_tokens, num_kv_heads, head_dim, device=DEVICE, dtype=dtype)
    v_bf16 = torch.randn(total_tokens, num_kv_heads, dv, device=DEVICE, dtype=dtype)

    cache_size = total_tokens
    loc = torch.arange(total_tokens, device=DEVICE, dtype=torch.int32)
    k_i4 = torch.zeros(cache_size, num_kv_heads, head_dim // 2, device=DEVICE, dtype=torch.uint8)
    v_i4 = torch.zeros(cache_size, num_kv_heads, dv // 2, device=DEVICE, dtype=torch.uint8)
    k_sz = torch.zeros(cache_size, num_kv_heads, 2, device=DEVICE, dtype=torch.float32)
    v_sz = torch.zeros(cache_size, num_kv_heads, 2, device=DEVICE, dtype=torch.float32)

    quantized_set_kv_int4_triton(k_bf16, v_bf16, loc, k_i4, v_i4, k_sz, v_sz)
    torch.cuda.synchronize()

    kv_indptr = torch.zeros(batch + 1, dtype=torch.int32, device=DEVICE)
    kv_indptr[1:] = torch.cumsum(
        torch.full((batch,), seq_len, device=DEVICE, dtype=torch.int32), dim=0
    )
    kv_indices = torch.arange(total_tokens, device=DEVICE, dtype=torch.int32)

    num_kv_splits = torch.full((batch,), kv_splits_val, dtype=torch.int32, device=DEVICE)
    attn_logits = torch.empty(
        batch, num_q_heads, max_kv_splits, dv, device=DEVICE, dtype=torch.float32
    )
    attn_lse = torch.empty(
        batch, num_q_heads, max_kv_splits, device=DEVICE, dtype=torch.float32
    )
    o = torch.zeros(batch, num_q_heads, dv, device=DEVICE, dtype=dtype)

    return dict(
        q=q,
        k_i4=k_i4,
        v_i4=v_i4,
        k_sz=k_sz,
        v_sz=v_sz,
        kv_indptr=kv_indptr,
        kv_indices=kv_indices,
        num_kv_splits=num_kv_splits,
        attn_logits=attn_logits,
        attn_lse=attn_lse,
        o=o,
        sm_scale=sm_scale,
        max_kv_splits=max_kv_splits,
    )


def run_decode_quantized(
    *,
    q: torch.Tensor,
    o: torch.Tensor,
    attn_logits: torch.Tensor,
    attn_lse: torch.Tensor,
    fuse_q_hadamard: bool,
    hadamard_order: int,
    **buffers,
):
    o.zero_()
    decode_attention_fwd_quantized(
        q,
        buffers["k_i4"],
        buffers["v_i4"],
        buffers["k_sz"],
        buffers["v_sz"],
        o,
        buffers["kv_indptr"],
        buffers["kv_indices"],
        attn_logits,
        attn_lse,
        buffers["num_kv_splits"],
        buffers["max_kv_splits"],
        buffers["sm_scale"],
        "int4",
        logit_cap=0.0,
        sinks=None,
        xai_temperature_len=-1,
        fuse_q_hadamard=fuse_q_hadamard,
        hadamard_order=hadamard_order,
    )
    torch.cuda.synchronize()


def sanity_fused_vs_unfused_q_hadamard(
    hadamard_order: int = 16,
    batch: int = 4,
    seq_len: int = 128,
    num_q_heads: int = 32,
    num_kv_heads: int = 8,
    head_dim: int = 128,
    atol: float = 0.08,
    rtol: float = 0.08,
) -> None:
    if head_dim % hadamard_order:
        raise ValueError(f"head_dim {head_dim} must be divisible by hadamard_order {hadamard_order}")

    bufs = build_gqa_int4_decode_buffers(
        batch, seq_len, num_q_heads, num_kv_heads, head_dim
    )
    q0 = bufs["q"].clone()

    o_ref = torch.zeros_like(bufs["o"])
    o_fused = torch.zeros_like(bufs["o"])
    logits_ref = torch.empty_like(bufs["attn_logits"])
    logits_fused = torch.empty_like(bufs["attn_logits"])
    lse_ref = torch.empty_like(bufs["attn_lse"])
    lse_fused = torch.empty_like(bufs["attn_lse"])

    q_ref = apply_triton_backend_style_q_hadamard(q0.clone(), hadamard_order)
    run_decode_quantized(
        q=q_ref,
        o=o_ref,
        attn_logits=logits_ref,
        attn_lse=lse_ref,
        fuse_q_hadamard=False,
        hadamard_order=hadamard_order,
        **{k: v for k, v in bufs.items() if k not in ("q", "o", "attn_logits", "attn_lse")},
    )

    run_decode_quantized(
        q=q0,
        o=o_fused,
        attn_logits=logits_fused,
        attn_lse=lse_fused,
        fuse_q_hadamard=True,
        hadamard_order=hadamard_order,
        **{k: v for k, v in bufs.items() if k not in ("q", "o", "attn_logits", "attn_lse")},
    )

    a = o_ref.float().flatten()
    b = o_fused.float().flatten()
    cos = torch.nn.functional.cosine_similarity(a, b, dim=0).item()
    max_abs = (o_ref.float() - o_fused.float()).abs().max().item()
    ok = torch.allclose(o_ref.float(), o_fused.float(), atol=atol, rtol=rtol)
    tag = f"order={hadamard_order} B={batch} S={seq_len} Hq={num_q_heads} Hkv={num_kv_heads} D={head_dim}"
    print(f"Sanity ({tag})")
    print(f"  cosine(o_ref, o_fused) = {cos:.6f}")
    print(f"  max_abs(o_ref - o_fused) = {max_abs:.6e}")
    print(f"  allclose(atol={atol}, rtol={rtol}): {ok}")
    assert cos >= 0.99, (
        f"Fused vs unfused outputs diverge (cosine={cos}); check dtypes / head_dim vs order."
    )


def build_mha_int4_decode_buffers(**kwargs):
    """MHA: same as GQA with H_kv = H_q."""
    h = kwargs.pop("num_heads")
    return build_gqa_int4_decode_buffers(
        num_q_heads=h, num_kv_heads=h, **kwargs
    )


def sanity_fused_vs_unfused_q_hadamard_mha(
    hadamard_order: int = 16,
    batch: int = 4,
    seq_len: int = 128,
    num_heads: int = 32,
    head_dim: int = 128,
    atol: float = 0.08,
    rtol: float = 0.08,
) -> None:
    if head_dim % hadamard_order:
        raise ValueError(f"head_dim {head_dim} must be divisible by hadamard_order {hadamard_order}")

    bufs = build_mha_int4_decode_buffers(
        batch=batch, seq_len=seq_len, num_heads=num_heads, head_dim=head_dim
    )
    q0 = bufs["q"].clone()
    o_ref = torch.zeros_like(bufs["o"])
    o_fused = torch.zeros_like(bufs["o"])
    logits_ref = torch.empty_like(bufs["attn_logits"])
    logits_fused = torch.empty_like(bufs["attn_logits"])
    lse_ref = torch.empty_like(bufs["attn_lse"])
    lse_fused = torch.empty_like(bufs["attn_lse"])

    q_ref = apply_triton_backend_style_q_hadamard(q0.clone(), hadamard_order)
    run_decode_quantized(
        q=q_ref,
        o=o_ref,
        attn_logits=logits_ref,
        attn_lse=lse_ref,
        fuse_q_hadamard=False,
        hadamard_order=hadamard_order,
        **{k: v for k, v in bufs.items() if k not in ("q", "o", "attn_logits", "attn_lse")},
    )
    run_decode_quantized(
        q=q0,
        o=o_fused,
        attn_logits=logits_fused,
        attn_lse=lse_fused,
        fuse_q_hadamard=True,
        hadamard_order=hadamard_order,
        **{k: v for k, v in bufs.items() if k not in ("q", "o", "attn_logits", "attn_lse")},
    )
    cos = torch.nn.functional.cosine_similarity(
        o_ref.float().flatten(), o_fused.float().flatten(), dim=0
    ).item()
    max_abs = (o_ref.float() - o_fused.float()).abs().max().item()
    ok = torch.allclose(o_ref.float(), o_fused.float(), atol=atol, rtol=rtol)
    tag = f"MHA order={hadamard_order} B={batch} S={seq_len} H={num_heads} D={head_dim}"
    print(f"Sanity ({tag})")
    print(f"  cosine(o_ref, o_fused) = {cos:.6f}")
    print(f"  max_abs(o_ref - o_fused) = {max_abs:.6e}")
    print(f"  allclose(atol={atol}, rtol={rtol}): {ok}")
    assert cos >= 0.99, f"MHA fused vs unfused diverge (cosine={cos})"


print("--- GQA ---")
for _order in (16, 64, 128):
    sanity_fused_vs_unfused_q_hadamard(hadamard_order=_order, head_dim=128)

print("--- MHA ---")
for _order in (16, 64, 128):
    sanity_fused_vs_unfused_q_hadamard_mha(hadamard_order=_order, head_dim=128)

Sanity (order=16 B=4 S=128 Hq=32 Hkv=8 D=128)
  cosine(o_ref, o_fused) = 1.000000
  max_abs(o_ref - o_fused) = 0.000000e+00
  allclose(atol=0.08, rtol=0.08): True
Sanity (order=64 B=4 S=128 Hq=32 Hkv=8 D=128)
  cosine(o_ref, o_fused) = 1.000000
  max_abs(o_ref - o_fused) = 0.000000e+00
  allclose(atol=0.08, rtol=0.08): True
Sanity (order=128 B=4 S=128 Hq=32 Hkv=8 D=128)
  cosine(o_ref, o_fused) = 0.999993
  max_abs(o_ref - o_fused) = 3.906250e-03
  allclose(atol=0.08, rtol=0.08): True


## 2. Throughput: GQA + MHA

Times **only** `decode_attention_fwd_quantized` (int4), using `triton.testing.do_bench` (ms). Same knobs for both; MHA uses `H == H_kv` so dispatch hits `_fwd_kernel_stage1_quant_int4`.

Rows:
- **no_q_hadamard:** `fuse_q_hadamard=False`, identity `q`.
- **fused_O:** `fuse_q_hadamard=True` with the listed order (FWHT inside the decode kernel).
- **unfused_cuda_q_then_decode:** for comparison, `hadamard_transform(q/sqrt(O))` in Python then `fuse_q_hadamard=False` (two-kernel style like MHA / non-fused GQA in `triton_backend`).

In [4]:
# --- Benchmark knobs (match your target model decode) ---
B = 8
SEQ_LEN = 2048
H_Q = 32
H_KV = 8
HEAD_DIM = 128
MAX_KV_SPLITS = 8
KV_SPLITS_VAL = 4
WARMUP = 15
REP = 50

DTYPE = torch.bfloat16
HADAMARD_ORDERS = (16, 64, 128)

assert HEAD_DIM % max(HADAMARD_ORDERS) == 0, "HEAD_DIM must be divisible by largest order"

base = build_gqa_int4_decode_buffers(
    B,
    SEQ_LEN,
    H_Q,
    H_KV,
    HEAD_DIM,
    dtype=DTYPE,
    max_kv_splits=MAX_KV_SPLITS,
    kv_splits_val=KV_SPLITS_VAL,
)


def _bench_int4_decode_for_base(base_local, q_fixed, label: str, hq: int, hkv: int):
    n_tok = B * SEQ_LEN
    bytes_q = B * hq * HEAD_DIM * 2
    bytes_kv_pack = n_tok * hkv * HEAD_DIM
    bytes_kv_sz = n_tok * hkv * 2 * 4 * 2
    bytes_o = B * hq * HEAD_DIM * 2
    bytes_logits = B * hq * MAX_KV_SPLITS * HEAD_DIM * 4
    bytes_lse = B * hq * MAX_KV_SPLITS * 4
    bt = bytes_q + bytes_kv_pack + bytes_kv_sz + bytes_o + bytes_logits + bytes_lse

    def clone_wb():
        return {
            "q": q_fixed.clone(),
            "o": torch.zeros_like(base_local["o"]),
            "attn_logits": torch.empty_like(base_local["attn_logits"]),
            "attn_lse": torch.empty_like(base_local["attn_lse"]),
            **{k: base_local[k] for k in (
                "k_i4", "v_i4", "k_sz", "v_sz",
                "kv_indptr", "kv_indices", "num_kv_splits", "sm_scale", "max_kv_splits",
            )},
        }

    def make_bf(*, kernel_fuse_q_hadamard: bool, hadamard_order: int, apply_cuda_q_hadamard_first: bool):
        def fn():
            w = clone_wb()
            q = w["q"]
            if apply_cuda_q_hadamard_first:
                q = apply_triton_backend_style_q_hadamard(q, hadamard_order)
            use_fuse = kernel_fuse_q_hadamard and not apply_cuda_q_hadamard_first
            run_decode_quantized(
                q=q,
                o=w["o"],
                attn_logits=w["attn_logits"],
                attn_lse=w["attn_lse"],
                fuse_q_hadamard=use_fuse,
                hadamard_order=hadamard_order,
                **{k: w[k] for k in (
                    "k_i4", "v_i4", "k_sz", "v_sz",
                    "kv_indptr", "kv_indices", "num_kv_splits",
                    "sm_scale", "max_kv_splits",
                )},
            )

        return fn

    print(f"\n=== {label} ===")
    print(f"  B={B} SEQ_LEN={SEQ_LEN} H_Q={hq} H_KV={hkv} HEAD_DIM={HEAD_DIM}")
    print(f"  ~traffic sum (approx): {bt / 1e6:.2f} MB / call\n")
    ms0 = triton.testing.do_bench(
        make_bf(kernel_fuse_q_hadamard=False, hadamard_order=16, apply_cuda_q_hadamard_first=False),
        warmup=WARMUP,
        rep=REP,
    )
    print(f"no_q_hadamard:              {ms0:.4f} ms  (~{bt / ms0 / 1e6:.2f} GB/s)")
    for O in HADAMARD_ORDERS:
        ms_f = triton.testing.do_bench(
            make_bf(kernel_fuse_q_hadamard=True, hadamard_order=O, apply_cuda_q_hadamard_first=False),
            warmup=WARMUP,
            rep=REP,
        )
        print(f"fused_q_hadamard O={O:3d}:     {ms_f:.4f} ms  (~{bt / ms_f / 1e6:.2f} GB/s)")
    for O in HADAMARD_ORDERS:
        ms_u = triton.testing.do_bench(
            make_bf(kernel_fuse_q_hadamard=False, hadamard_order=O, apply_cuda_q_hadamard_first=True),
            warmup=WARMUP,
            rep=REP,
        )
        print(f"unfused (CUDA q + decode) O={O:3d}: {ms_u:.4f} ms  (~{bt / ms_u / 1e6:.2f} GB/s)")


_bench_int4_decode_for_base(
    base, base["q"].clone(), "Decode int4 GQA benchmark", H_Q, H_KV
)

base_mha = build_mha_int4_decode_buffers(
    batch=B,
    seq_len=SEQ_LEN,
    num_heads=H_Q,
    head_dim=HEAD_DIM,
    dtype=DTYPE,
    max_kv_splits=MAX_KV_SPLITS,
    kv_splits_val=KV_SPLITS_VAL,
)
_bench_int4_decode_for_base(base_mha, base_mha["q"].clone(), "Decode int4 MHA benchmark", H_Q, H_Q)

print("\nDone.")

=== Decode int4 GQA benchmark ===
  B=8 SEQ_LEN=2048 H_Q=32 H_KV=8 HEAD_DIM=128
  ~traffic sum (approx): 20.06 MB / call

no_q_hadamard:              0.0751 ms  (~267.23 GB/s vs rough touch sum)
fused_q_hadamard O= 16:     0.0754 ms  (~266.14 GB/s)
fused_q_hadamard O= 64:     0.0754 ms  (~266.22 GB/s)
fused_q_hadamard O=128:     0.0754 ms  (~266.03 GB/s)
unfused (CUDA q + decode) O= 16: 0.1171 ms  (~171.37 GB/s)
unfused (CUDA q + decode) O= 64: 0.1169 ms  (~171.55 GB/s)
unfused (CUDA q + decode) O=128: 0.1190 ms  (~168.54 GB/s)

Done.
